In [29]:
from pathlib import Path
import pandas as pd
from pathlib import Path
import datetime
from loguru import logger
import numpy as np
import copy
from typing import Literal

from phd_visualizations.regression import regression_plot
from phd_visualizations.calculations import SupportedInstruments 
from phd_visualizations import save_figure

import combined_cooler
import matlab

%reload_ext autoreload
%autoreload 2

data_path: Path = Path("../data/")
model_type: Literal["data", "physical"] = "data"

cc_model = combined_cooler.initialize()


In [30]:
df = pd.read_csv(data_path / "wct_out_exp.csv")
df


,Tamb,HR,Tin,q,w_fan,Tout,m_w_lost
0,17.49,23.11,37.39,11.83,21.00,30.24,127.68
1,23.82,55.77,37.50,11.95,21.00,31.93,102.45
2,16.91,28.45,36.13,11.83,25.79,28.00,141.43
3,29.23,17.30,37.15,8.99,27.45,27.99,143.71
4,22.84,17.08,39.42,11.95,21.00,31.81,142.11
5,23.57,63.84,40.27,12.10,24.15,33.03,134.58
6,15.66,13.58,39.74,11.66,21.00,30.89,150.68
7,29.26,31.06,40.72,8.97,48.00,26.03,221.52
8,15.79,16.14,35.35,18.05,21.00,30.81,123.87
9,31.04,22.37,32.90,17.78,44.91,26.98,186.25


In [31]:
model_func = cc_model.wct_model_data if model_type == "data" else cc_model.wct_model_physical

results: list[dict] = []
Twct_out = []
Ce_wct = []
Cw_wct = []
for idx, ds in df.iterrows():

    qwct_m3h = matlab.double([ds["q"]])
    wwct = matlab.double([ds["w_fan"]])
    Tamb_C = matlab.double([ds["Tamb"]])
    Tin_C = matlab.double([ds["Tin"]])
    HR_pp = matlab.double([ds["HR"]])

    options = {
        "raise_error_on_invalid_inputs": False,
        "model_data_path": "/workspaces/SOLhycool/modeling/data/models_data/model_data_wct_fp_pilot_plant_gaussian_cascade.mat",
        "lb": matlab.double([0.1,0.1, 5.0,5.0,0.]),
        "ub": matlab.double([50.0,99.99,55.0,24.8400,95.]),
        "silence_warnings": False,
        "ce_coeffs": matlab.double([[0.4118, -11.54, 189.4]]),
        "c_poppe": 1.52,
        "n_poppe": -0.69,
        "Ta_out": 40.0,
        "HR2": 100.0,
    }

    twct_out, ce_wct, cw_wct = model_func(Tamb_C, HR_pp, Tin_C, qwct_m3h, wwct, options, nargout=3)
    Twct_out.append(twct_out)
    Ce_wct.append(ce_wct)
    Cw_wct.append(cw_wct)
    
results_df = pd.DataFrame({
    "q": df["q"],
    "w_fan": df["w_fan"],
    "Tamb": df["Tamb"],
    "Tin": df["Tin"],
    "Tout": Twct_out,
    # "Ce_wct": Ce_wct,
    "m_w_lost": Cw_wct, 
})

results_df


,q,w_fan,Tamb,Tin,Tout,m_w_lost
0,11.83,21.00,17.49,37.39,29.599739,130.949304
1,11.95,21.00,23.82,37.50,31.690471,101.021195
2,11.83,25.79,16.91,36.13,27.677419,154.971705
3,8.99,27.45,29.23,37.15,28.032761,135.567483
4,11.95,21.00,22.84,39.42,31.461241,138.565288
5,12.10,24.15,23.57,40.27,32.692003,136.727072
6,11.66,21.00,15.66,39.74,30.127745,151.392961
7,8.97,48.00,29.26,40.72,26.101519,244.575482
8,18.05,21.00,15.79,35.35,30.260706,143.814800
9,17.78,44.91,31.04,32.90,27.248335,195.617534


In [32]:
fig = regression_plot(
    df_ref= df, 
    df_mod= results_df,
    # df_ref_bg= df_cal,
    # df_mod_bg= results_cal_df,
    # bg_label="Cal.",
    var_ids=["Tout", "m_w_lost"], 
    units=["℃", "l/h"], 
    instruments=[SupportedInstruments.pt100, SupportedInstruments.vortex_flow_meter],
    width=520, height=750,
    show_error_metrics=["r2", "mae"],
    inline_error_metrics_text=True,
    legend_pos="top_spaced",
    vertical_spacing=0.15,
    legend_y=1.05,
    title_y=0.98,
    var_labels=["Outlet temperature", "Water consumption"],
    title_text=f'<span style="color:#9573a6; font-weight:bold;">WCT</span> {model_type} model validation'
)

fig


In [26]:
save_figure(
    fig,
    figure_name=f"wct_pilot_plant_model_regression_{model_type}",
    figure_path=Path("../results"),
    formats=["png", "svg"],
)


2025-09-12 08:17:39.558 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/wct_pilot_plant_model_regression_data.png
2025-09-12 08:17:39.647 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/wct_pilot_plant_model_regression_data.svg


## Scaled model validation

In [34]:
df = pd.read_csv("/workspaces/SOLhycool/modeling/results/model_inputs_sampling/andasol_90MW/wct_out.csv")
df


,Tamb,HR,Tin,q,w_fan,Tout,m_w_lost,Ce
0,3.0,99.900000,26.25,1107.000000,64.589981,12.115012,17071.282492,185.285245
1,3.0,99.900000,26.25,1707.631579,47.284500,15.215790,21271.497875,114.284938
2,3.0,99.900000,26.25,2308.263158,40.692902,17.186025,24129.902224,94.791055
3,3.0,99.900000,26.25,2458.421053,70.184478,17.186025,25699.608556,214.479889
4,3.0,99.900000,26.25,2908.894737,37.168050,18.551492,26214.164193,86.025914
...,...,...,...,...,...,...,...,...
932,50.0,54.625129,43.00,3359.368421,21.672252,19.577148,92777.029436,75.383994
933,50.0,54.625129,43.00,3509.526316,34.967219,19.577148,96924.000437,100.947470
934,50.0,54.625129,43.00,3659.684211,50.854906,19.577148,101070.971437,157.481261
935,50.0,54.625129,43.00,3809.842105,71.959866,19.577148,105217.942435,279.199270


In [ ]:
assert model_type == "data", "Only data-driven model is available for this dataset"
model_func = cc_model.wct_model_data if model_type == "data" else cc_model.wct_model_physical

results: list[dict] = []
Twct_out = []
Ce_wct = []
Cw_wct = []
for idx, ds in df.iterrows():

    qwct_m3h = matlab.double([ds["q"]])
    wwct = matlab.double([ds["w_fan"]])
    Tamb_C = matlab.double([ds["Tamb"]])
    Tin_C = matlab.double([ds["Tin"]])
    HR_pp = matlab.double([ds["HR"]])

    options = {
        "raise_error_on_invalid_inputs": False,
        "model_data_path": "/workspaces/SOLhycool/modeling/data/models_data/model_data_wct_fp_andasol_gaussian_cascade.mat",
        "lb": matlab.double([0.1, 0.1, 5.0, 1100., 0.]),
        "ub": matlab.double([50.0, 99.99, 55.0, 4000., 95.]),
        # "ce_coeffs": matlab.double([[0.4118, -11.54, 189.4]]),
        # "c_poppe": 1.52,
        # "n_poppe": -0.69,
        # "Ta_out": 40.0,
        # "HR2": 100.0,
    }

    twct_out, ce_wct, cw_wct = model_func(Tamb_C, HR_pp, Tin_C, qwct_m3h, wwct, options, nargout=3)
    Twct_out.append(twct_out)
    Ce_wct.append(ce_wct)
    Cw_wct.append(cw_wct)
    
results_df = pd.DataFrame({
    "q": df["q"],
    "w_fan": df["w_fan"],
    "Tamb": df["Tamb"],
    "Tin": df["Tin"],
    "Tout": Twct_out,
    # "Ce_wct": Ce_wct,
    "m_w_lost": Cw_wct, 
})

results_df


,q,w_fan,Tamb,Tin,Tout,m_w_lost
0,1107.000000,64.589981,3.0,26.25,-25.554792,26.994407
1,1707.631579,47.284500,3.0,26.25,-25.554792,26.994407
2,2308.263158,40.692902,3.0,26.25,-25.554792,26.994407
3,2458.421053,70.184478,3.0,26.25,-25.554792,26.994407
4,2908.894737,37.168050,3.0,26.25,-25.554792,26.994407
...,...,...,...,...,...,...
932,3359.368421,21.672252,50.0,43.00,-25.554792,26.994407
933,3509.526316,34.967219,50.0,43.00,-25.554792,26.994407
934,3659.684211,50.854906,50.0,43.00,-25.554792,26.994407
935,3809.842105,71.959866,50.0,43.00,-25.554792,26.994407


In [ ]:

fig = regression_plot(
    df_ref= df, 
    df_mod= results_df,
    df_ref_bg= df_cal,
    df_mod_bg= results_cal_df,
    bg_label="Cal.",
    var_ids=["Tout"], 
    units=["℃"], 
    instruments=[SupportedInstruments.pt100],
    width=520, height=750//2,
    show_error_metrics=["r2", "mae"],
    inline_error_metrics_text=True,
    legend_pos="top_spaced",
    vertical_spacing=0.15,
    legend_y=1.15,
    title_y=0.98,
    var_labels=["Outlet temperature"],
    title_text='<span style="color:#9573a6; font-weight:bold;">DC</span> model validation'
)

fig


In [ ]:
save_figure(
    fig,
    figure_name="dc_model_regression",
    figure_path=Path("../results"),
    formats=["png", "svg"],
)
